<a href="https://colab.research.google.com/github/daviseemann/turbofan-rul-prediction-cmapss/blob/production/notebooks/autoencoder-0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
drive.mount('/content/drive/', )
import os
os.chdir('/content/drive/MyDrive/Data science studies/Aprendizado-de-maquina-UFSC/final-project/data')

# Importando os dados

In [ ]:
# Caminhos dos arquivos
train_path = "./6.turbofan rul/train_FD001.txt"
test_path = "./6.turbofan rul/test_FD001.txt"
rul_path = "./6.turbofan rul/RUL_FD001.txt"

# Nomes das colunas (de acordo com a documentação original do C-MAPSS)
column_names = ['engine_id', 'cycle'] + \
               [f'op{i}' for i in range(1, 4)] + \
               [f's{i}' for i in range(1, 22)]

# Importando os arquivos (espaço em branco como delimitador)
df_train = pd.read_csv(train_path, sep='\s+', header=None, names=column_names)
df_test = pd.read_csv(test_path, sep='\s+', header=None, names=column_names)
df_rul = pd.read_csv(rul_path, sep='\s+', header=None, names=['RUL'])

In [ ]:
df_train.head()

# Tratamento dos dados

## Escalar os dados usando o MinMaxScalar

Importante se atentar aqui que o scaler foi desenvolvido para fazer o tratamento dos dados segmentados pelo motor, evitando assim um data leakege.

In [ ]:
from sklearn.preprocessing import MinMaxScaler


def scale_per_engine(df, feature_cols):
    """Aplica MinMaxScaler para cada motor individualmente."""
    df_scaled = df.copy()
    for eid in df['engine_id'].unique():
        idx = df['engine_id'] == eid
        scaler = MinMaxScaler()
        df_scaled.loc[idx, feature_cols] = scaler.fit_transform(df.loc[idx, feature_cols])
    return df_scaled

## Fazendo o calculo do RUL

Isso foi definido baseado no paper de referencia da MLP que usa um conceito mostrado no paper oficial do dataset. Onde estima-se um RUL máximo sobre o motor onde está considerando o comportamento "normal".

In [ ]:
# Cálculo da vida útil
def compute_rul(df, max_rul=125):
    """Calcula o RUL com limite superior (clipping)."""
    df = df.copy()
    max_cycle = df.groupby('engine_id')['cycle'].transform('max')
    df['RUL'] = (max_cycle - df['cycle']).clip(upper=max_rul)
    return df

## Fazendo a divisão dos dados em janelas

Uma MLP não consegue entender dados sequenciais de forma precisa, então para resolver esse problema foi definida a arquitetura de janelas. Desta forma, n_w amostras são empacotadas em um vetor e atribuida a label do rul de seu ultimo ciclo. Isso ajuda a MLP a compreender os dados de maneira mais precisa fazendo uma estimativa mais congruente com a natureza do problema.

In [ ]:
def generate_windows(df, sequence_length=30, features=None):
    """Gera janelas temporais com flattened input e target RUL."""
    X, y = [], []
    for eid in df['engine_id'].unique():
        engine_df = df[df['engine_id'] == eid].sort_values('cycle')
        sensor_data = engine_df[features].values
        rul_data = engine_df['RUL'].values

        for i in range(len(engine_df) - sequence_length + 1):
            window = sensor_data[i:i+sequence_length].flatten()
            target = rul_data[i + sequence_length - 1]
            X.append(window)
            y.append(target)

    return np.array(X), np.array(y)


In [ ]:
def plot_window_multiple_features(df, engine_id, feature_list, sequence_length=30):
    """
    Plota as janelas temporais em vários sensores, empilhados em uma única coluna.

    Args:
        df (pd.DataFrame): dataframe original com dados e 'engine_id'.
        engine_id (int): ID do motor a ser analisado.
        feature_list (list): lista de colunas sensoriais.
        sequence_length (int): tamanho da janela deslizante.
    """
    series_df = df[df['engine_id'] == engine_id].reset_index(drop=True)
    n = len(feature_list)

    fig, axes = plt.subplots(n, 1, figsize=(10, 2 * n), sharex=True)

    if n == 1:
        axes = [axes]

    for i, feature in enumerate(feature_list):
        ax = axes[i]
        ax.plot(series_df[feature], label=feature)
        for j in range(3):
            ax.axvspan(j, j + sequence_length, color='orange', alpha=0.3 if j == 0 else 0.15)
        ax.set_ylabel(feature)
        ax.grid(True)
        ax.legend(loc='upper right')

    axes[-1].set_xlabel("Ciclo")
    fig.suptitle(f'Visualização das Janelas Temporais - Motor {engine_id}', fontsize=14)
    plt.tight_layout()
    plt.show()


## Divisão entre treino e teste

Importante se atentar em não ocorrer data leakage, assim serão dividios os dados de forma que nenhum motor apareça no conjunto de teste e validação. Evitando um possível data leakege atrapalhando nosso controle sobre o treino e interpretação dos resultados.

In [ ]:
from sklearn.model_selection import train_test_split

def split_by_engine(df, test_size=0.2, random_state=42):
    """Divide o dataset com base nos engine_ids (sem vazamento temporal)."""
    engine_ids = df['engine_id'].unique()
    train_ids, val_ids = train_test_split(engine_ids, test_size=test_size, random_state=random_state)

    df_train = df[df['engine_id'].isin(train_ids)].reset_index(drop=True)
    df_val = df[df['engine_id'].isin(val_ids)].reset_index(drop=True)

    return df_train, df_val

# Pipeline de tratamento dos dados

In [ ]:
def prepare_data_for_mlp(df, sequence_length=30, max_rul=125, test_size=0.2, random_state=42):
    """
    Pipeline completa de preparação dos dados para MLP.
    Retorna X_train, y_train, X_val, y_val prontos para treinamento.
    """
    # 1. Definir colunas de sensores (assumindo prefixo 's')
    feature_cols = [col for col in df.columns if col.startswith('s')]

    # 2. Calcular RUL com limite superior
    df_rul = compute_rul(df.copy(), max_rul=max_rul)

    # 3. Escalar sensores individualmente por motor
    df_scaled = scale_per_engine(df_rul, feature_cols)

    # 4. Separar por motores (sem data leakage)
    df_train, df_val = split_by_engine(df_scaled, test_size=test_size, random_state=random_state)

    # 5. Gerar janelas temporais para MLP
    X_train, y_train = generate_windows(df_train, sequence_length=sequence_length, features=feature_cols)
    X_val, y_val = generate_windows(df_val, sequence_length=sequence_length, features=feature_cols)

    return X_train, y_train, X_val, y_val

In [ ]:
feature_cols = [col for col in df_train.columns if col.startswith('s')]
plot_window_multiple_features( df_train,engine_id=1, feature_list=feature_cols[:5], sequence_length=30)

# Construção do modelo MLP

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow.keras.backend as K

In [ ]:
def build_mlp(input_dim):
    """Cria modelo MLP baseado no paper."""
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(100, activation='relu'),
        Dense(50, activation='relu'),
        Dense(1)  # saída: RUL
    ])

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='mse',
        metrics=[rmse_metric, nasa_score_metric]
    )

    return model


In [ ]:
def rmse_metric(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def nasa_score_metric(y_true, y_pred):
    diff = y_pred - y_true
    score = K.switch(
        K.greater(diff, 0),
        K.exp(diff / 13.0) - 1,
        K.exp(-diff / 10.0) - 1
    )
    return K.mean(score)

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [ ]:
model = build_mlp(X.shape[1])
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)